# EDS v1 Processing Walkthrough

This notebook is a compact, didactic view of the Energi Data Service v1 price-processing contract.

It intentionally uses tiny source-shaped examples instead of replaying every implementation detail from the production pipeline. The goal is to make the important judgement calls visible:

- preserve raw API responses as audit evidence;
- normalize old and new EDS price schemas into one canonical shape;
- validate source local time against UTC-derived Danish time;
- stitch `Elspotprices` to `DayAheadPrices` at the 2025-10-01 local delivery boundary;
- aggregate quarter-hour prices to hourly prices only when the hour is complete;
- enforce the official DK1/DK2 model-ready panel shape.

Canonical implementation references:

- `src/dkenergy_data/sources/energidataservice.py`
- `src/dkenergy_data/build/eds_prices_v1.py`
- `docs/data-processing/energi_data_service_v1.md`


In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
import tempfile
from datetime import date, datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except ImportError:
    sns = None
    plt.style.use("seaborn-v0_8-whitegrid")


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root with pyproject.toml and src/")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dkenergy_data.build.eds_prices_v1 import (
    ALLOWED_AREAS,
    DATASET_VERSION,
    PANEL_COLUMNS,
    SOURCE_SPECS,
    STITCH_BOUNDARY_DATE,
    STITCH_BOUNDARY_LOCAL,
    RawBatch,
    build_model_ready_panel,
    normalize_batches,
)
from dkenergy_data.sources.energidataservice import ApiResponse, write_dataset_response

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset version: {DATASET_VERSION}")
print(f"Official v1 areas: {', '.join(ALLOWED_AREAS)}")
print(f"Stitch boundary: {STITCH_BOUNDARY_LOCAL}")


## 1. Source contract at a glance

EDS exposes the historical and current price data through two related datasets. The production pipeline treats that source transition as part of the dataset definition, not as an incidental implementation detail.


In [ ]:
schema_rows = []
for name, spec in SOURCE_SPECS.items():
    schema_rows.append(
        {
            "source_dataset": name,
            "utc_time_column": spec.time_column,
            "source_local_column": spec.local_time_column,
            "area_column": spec.area_column,
            "dkk_column": spec.price_dkk_column,
            "eur_column": spec.price_eur_column,
            "native_resolution_minutes": spec.source_resolution_minutes,
        }
    )

pd.DataFrame(schema_rows)


## 2. Raw audit evidence is byte-preserved

The raw layer is not a cleaned JSON representation. It is the HTTP response body saved exactly as received. The manifest stores both the response hash and the saved-file hash so a later audit can prove the raw file was not changed during save.


In [ ]:
raw_bytes = b'{"dataset":"DayAheadPrices","records":[{"TimeUTC":"2025-09-30T22:00:00"}]}'
response = ApiResponse(
    url="https://api.energidataservice.dk/dataset/DayAheadPrices",
    params={"limit": 0},
    status_code=200,
    content=raw_bytes,
    payload=json.loads(raw_bytes),
    retrieved_at_utc=datetime(2026, 1, 1, tzinfo=timezone.utc),
    response_sha256=hashlib.sha256(raw_bytes).hexdigest(),
)

with tempfile.TemporaryDirectory() as tmp:
    result = write_dataset_response(
        Path(tmp),
        "DayAheadPrices",
        date(2025, 10, 1),
        date(2025, 10, 2),
        response,
    )
    saved_bytes = result.path.read_bytes()
    audit_check = pd.DataFrame(
        [
            {
                "check": "saved bytes equal response bytes",
                "value": saved_bytes == raw_bytes,
            },
            {
                "check": "response_sha256 equals saved_json_sha256",
                "value": result.manifest_entry["response_sha256"]
                == result.manifest_entry["saved_json_sha256"],
            },
        ]
    )

audit_check


## 3. A tiny source-shaped boundary fixture

The example below has two old hourly `Elspotprices` rows per area and one new hourly value per area after quarter-hour aggregation. This is deliberately tiny: enough to show the logic without hiding it in a large dataframe.


In [ ]:
def make_boundary_batches() -> tuple[list[RawBatch], list[RawBatch]]:
    elspot_records = []
    for area, base in [("DK1", 100.0), ("DK2", 110.0)]:
        elspot_records.extend(
            [
                {
                    "HourUTC": "2025-09-30T20:00:00",
                    "HourDK": "2025-09-30T22:00:00",
                    "PriceArea": area,
                    "SpotPriceDKK": base,
                    "SpotPriceEUR": base / 7.45,
                },
                {
                    "HourUTC": "2025-09-30T21:00:00",
                    "HourDK": "2025-09-30T23:00:00",
                    "PriceArea": area,
                    "SpotPriceDKK": base + 20.0,
                    "SpotPriceEUR": (base + 20.0) / 7.45,
                },
            ]
        )

    day_ahead_records = []
    quarter_hours = ["22:00:00", "22:15:00", "22:30:00", "22:45:00"]
    local_times = ["00:00:00", "00:15:00", "00:30:00", "00:45:00"]
    for area, values in [("DK1", [200, 220, 240, 260]), ("DK2", [210, 230, 250, 270])]:
        for utc_time, local_time, value in zip(quarter_hours, local_times, values):
            day_ahead_records.append(
                {
                    "TimeUTC": f"2025-09-30T{utc_time}",
                    "TimeDK": f"2025-10-01T{local_time}",
                    "PriceArea": area,
                    "DayAheadPriceDKK": float(value),
                    "DayAheadPriceEUR": float(value) / 7.45,
                }
            )

    return [RawBatch("Elspotprices", "old_demo", "2026-01-01T00:00:00+00:00", Path("demo"), elspot_records)], [
        RawBatch("DayAheadPrices", "new_demo", "2026-01-01T00:00:00+00:00", Path("demo"), day_ahead_records)
    ]


elspot_batches, day_ahead_batches = make_boundary_batches()
elspot_normalized = normalize_batches(elspot_batches)
day_ahead_normalized = normalize_batches(day_ahead_batches)

pd.concat([elspot_normalized.head(4), day_ahead_normalized.head(4)], ignore_index=True)


## 4. Source local timestamps are validated, not trusted blindly

UTC is the canonical key. The source local timestamp is still valuable: it catches source-schema or DST surprises. If EDS says a UTC hour maps to a different Danish wall-clock time than `Europe/Copenhagen` says, normalization fails.


In [ ]:
bad_batch = RawBatch(
    source_dataset="Elspotprices",
    batch_id="bad_local_time_demo",
    retrieved_at_utc="2026-01-01T00:00:00+00:00",
    raw_path=Path("demo"),
    records=[
        {
            "HourUTC": "2025-09-30T20:00:00",
            "HourDK": "2025-09-30T23:00:00",  # should be 22:00 in Copenhagen
            "PriceArea": "DK1",
            "SpotPriceDKK": 100.0,
            "SpotPriceEUR": 13.4,
        }
    ],
)

try:
    normalize_batches([bad_batch])
except ValueError as exc:
    print(str(exc).split(":", 1)[0])


## 5. Stitching and quarter-hour aggregation

The official hourly target is defined by Danish local delivery time:

- `Elspotprices` before `2025-10-01 00:00` local;
- `DayAheadPrices` from `2025-10-01 00:00` local onward.

For the new quarter-hour source, exactly four 15-minute rows are required per area-hour. The hourly price is the arithmetic mean of those four rows.


In [ ]:
panel, qa = build_model_ready_panel(elspot_normalized, day_ahead_normalized)

panel[["unique_id", "ds_utc", "ds_local", "area", "y", "source_dataset", "source_resolution_minutes"]]


In [ ]:
qa_view = pd.DataFrame(
    [
        {"field": "artifact_status", "value": qa["artifact_status"]},
        {"field": "areas", "value": qa["areas"]},
        {"field": "row_count", "value": qa["row_count"]},
        {"field": "missing_hour_count", "value": qa["missing_hour_count"]},
        {"field": "transition_boundary_check", "value": qa["transition_boundary_check"]["status"]},
        {"field": "shared_utc_coverage_check", "value": qa["shared_utc_coverage_check"]["status"]},
    ]
)
qa_view


In [ ]:
plot_frame = panel.copy()
plot_frame["ds_local_label"] = plot_frame["ds_local"].dt.strftime("%Y-%m-%d %H:%M")
fig, ax = plt.subplots(figsize=(9, 4))
for area, area_frame in plot_frame.groupby("area"):
    ax.plot(area_frame["ds_local"], area_frame["y"], marker="o", label=area)
ax.axvline(STITCH_BOUNDARY_LOCAL, color="black", linestyle="--", linewidth=1, label="source boundary")
ax.set_title("Tiny boundary fixture: old hourly source stitched to new quarter-hour source")
ax.set_xlabel("Danish local delivery time")
ax.set_ylabel("DKK/MWh")
ax.legend()
fig.autofmt_xdate()
plt.show()


## 6. The official panel is two-area by default

The forecasting contract expects an hourly panel per `unique_id`, and the project target is DK1/DK2. Official v1 builds therefore require both price areas and identical UTC coverage. A one-area panel is allowed only when the caller explicitly asks for an experiment.


In [ ]:
one_area_panel_input = panel.loc[panel["area"] == "DK1"].copy()
elspot_one_area = elspot_normalized.loc[elspot_normalized["area"] == "DK1"].copy()
day_ahead_one_area = day_ahead_normalized.loc[day_ahead_normalized["area"] == "DK1"].copy()

try:
    build_model_ready_panel(elspot_one_area, day_ahead_one_area)
except ValueError as exc:
    print("Default official build rejects one-area panels:")
    print(str(exc))

experiment_panel, experiment_qa = build_model_ready_panel(
    elspot_one_area,
    day_ahead_one_area,
    required_areas=["DK1"],
)
print("Explicit one-area experiment status:", experiment_qa["shared_utc_coverage_check"]["status"])


## 7. What the model-ready table promises

The forecasting side should not call EDS. It receives a model-ready parquet table with stable keys, calendar features, audit fields, and no silent target repairs.


In [ ]:
columns_table = pd.DataFrame(
    {
        "column": PANEL_COLUMNS,
        "role": [
            "forecast unit",
            "primary UTC time key",
            "Danish local display/feature time",
            "calendar feature",
            "calendar feature",
            "calendar feature",
            "calendar feature",
            "calendar feature",
            "DST feature",
            "DST disambiguation",
            "price area",
            "target, DKK/MWh",
            "target copy, DKK/MWh",
            "audit value, EUR/MWh",
            "audit source",
            "audit source resolution",
            "dataset version",
        ],
    }
)
columns_table


## 8. Canonical commands

This notebook teaches the processing choices. The canonical build remains the CLI/package pipeline:

```bash
python scripts/fetch_eds_prices.py --areas DK1 DK2
python scripts/build_price_panel.py --dataset-version v1
```

For a boundary smoke test:

```bash
python scripts/fetch_eds_prices.py --start 2025-09-29 --end 2025-10-03 --areas DK1 DK2
python scripts/build_price_panel.py --start 2025-09-29 --end 2025-10-03
```
